# Homework — Working with data (pandas)

**pandas** is the tool chemists reach for whenever data lives in a table: samples in
rows, measurements in columns. This homework teaches you to build a table, look at it,
compute new columns, filter it, sort it, and summarize it — and, just as important, to
*reason* about what pandas is doing.

**How this works**
- Run every cell top to bottom. Run the **setup cell** first or nothing else will work.
- Teaching cells already have the answer as a `# ...` comment — run them, then predict-then-check.
- Answer the numbered **Problems** in the empty cells. About half ask you to *think*, not just type.
- The **Check in** at the very end prints one value — type it back into the CHEM 438 app to finish.

In [ ]:
# Setup — run this first
!pip install pandas -q
import pandas as pd
print("pandas is ready:", pd.__version__)

## 1. A table from a dictionary

A **DataFrame** is a table. The easiest way to make one is from a dictionary: each
**key becomes a column name**, and each **list becomes that column's values**. All the
lists must be the same length (one entry per row).

Once you have a DataFrame, four tools tell you what you're holding:
- `df.head(n)` — the first `n` rows (a peek, not the whole thing)
- `df.shape` — `(rows, columns)`
- `df.columns` — the column names
- `df.dtypes` — the *type* stored in each column (text vs. integer vs. decimal)

In [ ]:
data = {
    "Compound": ["Water", "Ethanol", "Acetone", "Benzene", "Glucose"],
    "MolarMass": [18.0, 46.1, 58.1, 78.1, 180.2],
    "Atoms": [3, 9, 10, 12, 24],
}
df = pd.DataFrame(data)

print(df.head(3))                     # first 3 rows only
print("shape:", df.shape)             # (5, 3)  -> 5 rows, 3 columns
print("columns:", list(df.columns))  # ['Compound', 'MolarMass', 'Atoms']
print(df.dtypes)                      # Compound=object(text), MolarMass=float64, Atoms=int64

**Problem 1 (predict, then run).** You just saw the compounds table `df`.
*Before running anything*, write down what you think `df.shape` returns and what
`len(df)` returns. Then in the cell below, run both and check.

In a comment, answer: **which number in `df.shape` equals `len(df)` — the rows or the
columns?**

## 2. One column vs. a list of columns

There are two ways to pull columns out, and they give **different kinds of object**:

- `df["MolarMass"]` — single brackets, one name → a **Series** (a single 1-D column)
- `df[["MolarMass"]]` — double brackets, a *list* of names → a **DataFrame** (a table that
  happens to have one column)

This matters constantly: a Series and a one-column DataFrame print differently, and many
functions expect one or the other. The rule to remember: **a list of column names always
gives you back a table.**

In [ ]:
col  = df["MolarMass"]     # single brackets -> Series
cols = df[["MolarMass"]]    # double brackets (a list!) -> DataFrame

print(type(col).__name__)   # Series
print(type(cols).__name__)  # DataFrame
print(col.shape)            # (5,)     one dimension
print(cols.shape)           # (5, 1)   two dimensions: 5 rows, 1 column

**Problem 2 (concept — multiple choice).** Based on section 2, which statement is
true about `df["MolarMass"]` versus `df[["MolarMass"]]`?

- **A.** They are identical — the extra brackets do nothing.
- **B.** `df["MolarMass"]` is a Series (1-D); `df[["MolarMass"]]` is a DataFrame (a table with one column).
- **C.** `df["MolarMass"]` is a DataFrame; `df[["MolarMass"]]` is a Series.
- **D.** `df[["MolarMass"]]` raises an error because you can't put a list in the brackets.

In the cell below, set `answer` to your choice (a string like `"A"`), then print it.

In [ ]:
answer = "?"   # replace ? with "A", "B", "C", or "D"
print(answer)

## 3. Computed columns and boolean masks

**Add a column** by assigning to a new name. Arithmetic on columns happens row by row:
`df["MolarMass"] / df["Atoms"]` divides each compound's mass by its own atom count.

**Filtering** works in two steps, and the first step is the one people skip over:
1. A comparison like `df["MolarMass"] > 50` does **not** return rows. It returns a
   **Series of True/False**, one per row — this is called a *boolean mask*.
2. Handing that mask back to the DataFrame, `df[mask]`, keeps only the rows where the
   mask is `True`.

So `df[df["MolarMass"] > 50]` is really `df[ <a column of True/False> ]`. Seeing the mask
by itself is the key to understanding filtering.

In [ ]:
df["MassPerAtom"] = df["MolarMass"] / df["Atoms"]   # new computed column
print(df[["Compound", "MassPerAtom"]].head(2))

mask = df["MolarMass"] > 50        # a column of True/False, NOT rows
print(mask.tolist())               # [False, False, True, True, True]

heavy = df[mask]                   # keep only rows where mask is True
print(heavy.shape)                 # (3, 4)  -> 3 rows survived, now 4 columns

**Problem 3 (why does this work?).** The filter `df[df["Atoms"] > 10]` works even
though it looks like you put a whole table comparison inside the brackets.

In the cell below: first print **just** `df["Atoms"] > 10` on its own. Look at what type
of thing it is and what it contains. Then print `df[df["Atoms"] > 10]`. In a comment,
explain in your own words **what the "thing inside the brackets" is** and why the
DataFrame knows which rows to keep.

**Problem 4 (read the error, fix the bug).** A classmate wrote the line below to pull
out the molar-mass column, but it crashes:

```python
df["Molar_Mass"]
```

Running it raises:

```
KeyError: 'Molar_Mass'
```

A `KeyError` means pandas looked for a column with that exact name and didn't find one.
In the cell below, write a version that **works**, and in a comment say exactly what was
wrong. (Look back at `df.columns` from section 1 if you need the real name.)

## 4. Sorting and summarizing

`df.sort_values("MolarMass", ascending=False)` returns a **new** DataFrame ordered by that
column (it does not change `df` itself).

To summarize numbers:
- `df["MolarMass"].mean()` and `.sum()` collapse a column to a single number.
- `df["MolarMass"].describe()` gives count, mean, std, min, the quartiles, and max all at
  once — a fast way to see the shape of your data before you plot anything.

In [ ]:
ordered = df.sort_values("MolarMass", ascending=False)
print(ordered["Compound"].tolist())   # ['Glucose', 'Benzene', 'Acetone', 'Ethanol', 'Water']

print("mean molar mass:", round(df["MolarMass"].mean(), 1))  # 76.1
print("total atoms    :", df["Atoms"].sum())                 # 58
print(df["MolarMass"].describe())      # count, mean, std, min, 25/50/75%, max

**Problem 5 (interpret the numbers).** From `df["MolarMass"].describe()` you can read
a **mean of about 76** but a **max of about 180**. The mean sits much closer to the
smaller values than to the max.

In the cell below, print `df["MolarMass"].describe()` again, then in a **comment** answer:
what does the large gap between the mean (~76) and the max (~180) suggest about the data?
Is the mean being pulled up by one unusually heavy compound, or are the values evenly
spread? Name the compound you think is responsible.

## 5. Reading a CSV file

Real data usually arrives as a **CSV** (comma-separated values) file, not a Python
dictionary. `pd.read_csv("filename.csv")` reads it straight into a DataFrame.

To keep this notebook self-contained, the cell below first *writes* a small CSV of lab
samples to disk, then reads it back — exactly the call you'd use on a file someone
handed you.

In [ ]:
csv_text = """Sample,Solvent,Mass_mg,Purity
S1,Water,12.5,98.2
S2,Ethanol,8.3,95.0
S3,Acetone,15.1,99.1
S4,Benzene,6.7,88.4
S5,Water,20.2,97.5
S6,Ethanol,9.9,91.3
"""
with open("samples.csv", "w") as f:
    f.write(csv_text)

samples = pd.read_csv("samples.csv")
print(samples.shape)       # (6, 4)
print(samples.head(2))

**Problem 6 (put it together).** Using the `samples` DataFrame from section 5:
1. Sort the samples by `Purity` from **highest to lowest** and print the result.
2. Print the **mean** `Mass_mg` across all samples.
3. Filter to only the rows where `Solvent` is `"Water"` and print that smaller table.
4. In a comment, state how many Water samples there are.

## Check in

Run the cell below. It reloads the samples file and counts how many samples have a
**Purity of at least 95**. It prints one number.

**Type that number into the CHEM 438 app to mark this homework complete.**

In [ ]:
samples = pd.read_csv("samples.csv")
high_purity = samples[samples["Purity"] >= 95]
print("Your check-in value:", high_purity.shape[0])